# Notebook Security and Usage Guardrails
- Do not store secrets, passwords, tokens, or keys in notebook source or outputs.
- Load credentials only from environment variables (`.env`) and keep outputs cleared before commit.
- Canonical production logic lives under `data_pipeline/src`; this notebook is for exploration/reference.


In [ ]:
import os
from pathlib import Path

REQUIRED_ENV_VARS = ["GOOGLE_CLIENT_ID", "GOOGLE_CLIENT_SECRET", "GOOGLE_TOKEN_PATH"]
missing = [name for name in REQUIRED_ENV_VARS if not os.getenv(name)]
if missing:
    raise ValueError(f"Missing required environment variable(s): {', '.join(missing)}")

project_root = Path.cwd()
if not (project_root / "data").exists():
    raise ValueError("Expected ./data directory at repository root; run notebook from repo root.")

print("Environment guard check passed.")

In [1]:
%config Completer.use_jedi = False
%load_ext autoreload

# Imports

In [ ]:
import json
import os
import pathlib
import re
import sqlite3
from urllib.parse import parse_qs, urlparse

import pandas as pd
from dotenv import load_dotenv
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

# Pull data from google sheet

In [ ]:
SCOPES = ["https://www.googleapis.com/auth/spreadsheets.readonly"]

In [ ]:
# Load Google OAuth client settings from .env.
CLIENT_ID = os.getenv("GOOGLE_CLIENT_ID")
CLIENT_SECRET = os.getenv("GOOGLE_CLIENT_SECRET")

if not CLIENT_ID:
    raise ValueError("GOOGLE_CLIENT_ID is not set in .env")

if not CLIENT_SECRET:
    raise ValueError("GOOGLE_CLIENT_SECRET is not set in .env")

In [ ]:
# DO NOT USE THIS ANYMORE USE THE NEXT CELL
# # Where to cache the user token so you don't need to re-auth each time
# TOKEN_PATH = pathlib.Path("token.json")

# # ====== 2) OAuth: load cached token or run local server flow ======
# creds = None
# if TOKEN_PATH.exists():
#     creds = Credentials.from_authorized_user_file(TOKEN_PATH, SCOPES)

# if not creds or not creds.valid:
#     if creds and creds.expired and creds.refresh_token:
#         # Try to refresh silently
#         try:
#             creds.refresh(request=None)  # google-auth refresh uses default request if None
#         except Exception:
#             creds = None

# if not creds or not creds.valid:
#     # Build an in-memory client config so we don't need a client_secret.json file
#     client_config = {
#         "installed": {
#             "client_id": CLIENT_ID,
#             "client_secret": CLIENT_SECRET,
#             "auth_uri": "https://accounts.google.com/o/oauth2/auth",
#             "token_uri": "https://oauth2.googleapis.com/token",
#             "redirect_uris": ["http://localhost"],
#         }
#     }
#     flow = InstalledAppFlow.from_client_config(client_config, SCOPES)
#     # Opens a browser window; after you approve, it returns to localhost and completes the flow
#     creds = flow.run_local_server(port=0)
#     #     if os.getenv("GOOGLE_WRITE_TOKEN_CACHE", "false").lower() == "true":
#         TOKEN_PATH.write_text(creds.to_json())
#     else:
#         print("Skipping token cache write (set GOOGLE_WRITE_TOKEN_CACHE=true to enable).")

In [ ]:
load_dotenv()  # loads .env from project root / current working dir

token_path_raw = os.getenv("GOOGLE_TOKEN_PATH")
if not token_path_raw:
    raise ValueError("GOOGLE_TOKEN_PATH is not set in .env")

CLIENT_ID = os.getenv("GOOGLE_CLIENT_ID")
if not CLIENT_ID:
    raise ValueError("GOOGLE_CLIENT_ID is not set in .env")

CLIENT_SECRET = os.getenv("GOOGLE_CLIENT_SECRET")
if not CLIENT_SECRET:
    raise ValueError("GOOGLE_CLIENT_SECRET is not set in .env")

TOKEN_PATH = pathlib.Path(token_path_raw).expanduser().resolve()
TOKEN_PATH.parent.mkdir(parents=True, exist_ok=True)

# ====== 2) OAuth: load cached token or run local server flow ======
creds = None
if TOKEN_PATH.exists():
    creds = Credentials.from_authorized_user_file(str(TOKEN_PATH), SCOPES)

if not creds or not creds.valid:
    if creds and creds.expired and creds.refresh_token:
        try:
            creds.refresh(Request())
        except Exception:
            creds = None

if not creds or not creds.valid:
    client_config = {
        "installed": {
            "client_id": CLIENT_ID,
            "client_secret": CLIENT_SECRET,
            "auth_uri": "https://accounts.google.com/o/oauth2/auth",
            "token_uri": "https://oauth2.googleapis.com/token",
            "redirect_uris": ["http://localhost"],
        }
    }
    flow = InstalledAppFlow.from_client_config(client_config, SCOPES)
    creds = flow.run_local_server(port=0)
    if os.getenv("GOOGLE_WRITE_TOKEN_CACHE", "false").lower() == "true":
        TOKEN_PATH.write_text(creds.to_json())
    else:
        print("Skipping token cache write (set GOOGLE_WRITE_TOKEN_CACHE=true to enable).")

In [ ]:
service = build("sheets", "v4", credentials=creds)

In [ ]:
SPREADSHEET_ID = "1bxd-VOEC_gQVq0E_apo2m81jRin0EdJFXXRzGxhS3KE"   
GID = 1509650705             
A1_RANGE = "A1:D1900"

In [ ]:
def list_tabs(service, spreadsheet_id):
    """Return [{title, sheetId, index}, ...]"""
    meta = service.spreadsheets().get(
        spreadsheetId=spreadsheet_id,
        fields="sheets(properties(title,sheetId,index))",
    ).execute()
    tabs = []
    for s in meta.get("sheets", []):
        p = s["properties"]
        tabs.append({"title": p["title"], "sheetId": p["sheetId"], "index": p.get("index", 0)})
    return tabs

def title_for_gid(tabs, gid):
    for t in tabs:
        if t["sheetId"] == gid:
            return t["title"]
    return None

In [ ]:
# 1) Resolve tab title from gid (sheetId)
tabs = list_tabs(service, SPREADSHEET_ID)
tab_title = title_for_gid(tabs, GID)
if tab_title is None:
    raise ValueError(f"gid {GID} not found in spreadsheet {SPREADSHEET_ID}")

In [ ]:
tab_title

In [ ]:
# 2) Read the range with title + A1
rng = f"'{tab_title}'!{A1_RANGE}"
resp = service.spreadsheets().values().get(
    spreadsheetId=SPREADSHEET_ID,
    range=rng
).execute()

values = resp.get("values", [])

In [ ]:
# 3) Optional: DataFrame with header row detection
if values:
    headers = values[0]
    rows = values[1:]
    if headers and all(h != "" for h in headers):
        df_sheet = pd.DataFrame(rows, columns=headers)
    else:
        df_sheet = pd.DataFrame(values)
else:
    df_sheet = pd.DataFrame()

In [ ]:
# for col in df.columns[1:]:
#     df_sheet[col] =   df_sheet[col].astype(int)
df_sheet = df_sheet.rename(columns = {"Board Game Titles":"name"})

In [ ]:
df_sheet.head()

In [ ]:
print(df_sheet.shape)

# Pull list of games from database

In [ ]:
conn = sqlite3.connect('../../backend/database/boardgames.db')
cursor = conn.cursor()
games_list = cursor.execute("SELECT id, name, num_ratings, year_published FROM games")

In [ ]:
rows = cursor.fetchall()

In [ ]:
df_games_list = pd.DataFrame(rows)

In [ ]:
df_games_list.columns = ["id", "name", "num_ratings", "year_published"]
df_games_list.head()

In [ ]:
df_games_list.shape

# Merge the two

## Match_v1

In [ ]:
df_merge_1 = df_games_list.merge(df_sheet, on="name", how="inner")
df_merge_1 = df_merge_1.sort_values(["num_ratings", "year_published"], ascending=False).drop_duplicates(subset="name")
df_merge_1.shape

In [ ]:
df_missing_1 = df_sheet.loc[~(df_sheet["name"].isin(df_merge_1["name"]))].copy()
df_missing_1.shape

## Match_v2

In [ ]:
df_games_list["name_lower"] = df_games_list["name"].str.lower()
df_missing_1["name_lower"] = df_missing_1["name"].str.lower()
df_merge_2 = df_games_list.drop(columns="name").merge(df_missing_1, on="name_lower", how="inner")
df_merge_2 = df_merge_2.sort_values(["num_ratings", "year_published"], ascending=False).drop_duplicates(subset="name_lower")
df_merge_2 = df_merge_2.drop(columns=["name_lower"])
df_merge_2.shape

In [ ]:
df_merge_2.head()

In [ ]:
df_merge = pd.concat([df_merge_1, df_merge_2])
print(df_merge.shape)
df_missing_2 = df_sheet.loc[~(df_sheet["name"].isin(df_merge["name"]))].copy()
df_missing_2.shape

## Match_v3

In [ ]:
df_manual = pd.read_csv("../../data/pax/pax_tt_games_1750556524.csv",sep="|")
df_manual = df_manual.dropna(subset=["bgg_id"])
df_manual["bgg_id"] = df_manual["bgg_id"].astype(int)
df_manual = df_manual.rename(columns={"name":"name_lower", "bgg_id":"id"})
df_manual = df_manual.loc[df_manual["name_raw"].isna()]
df_manual.head()

In [ ]:
df_missing_2["name_lower"] = df_missing_2["name"].str.lower()
df_merge_3 = df_manual[["name_lower", "id"]].merge(df_missing_2, on="name_lower", how="inner")
df_merge_3 = df_merge_3.drop(columns=["name_lower"])
df_merge_3 = df_merge_3.merge(df_games_list[["id", "num_ratings", "year_published"]], on="id")
df_merge_3 = df_merge_3[df_merge.columns.tolist()]
df_merge_3.shape

In [ ]:
df_merge = pd.concat([df_merge_1, df_merge_2, df_merge_3])
print(df_merge.shape)
df_missing_3 = df_sheet.loc[~(df_sheet["name"].isin(df_merge["name"]))].copy()
df_missing_3.shape